<a href="https://colab.research.google.com/github/AmeliaReginaa/PROJECT-AKHIR-DEEP-LEARNING-SISTEM-MULTIMEDIA/blob/main/Preprocesing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Preprocesing

In [ ]:
# 1. IMPORT LIBRARY
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
# Install Sastrawi if not already installed
!pip install Sastrawi
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
# 2. LOAD DATASET
# Load dataset
df_gojek = pd.read_csv('/content/gojek_reviews.csv')
df_grab = pd.read_csv('/content/grab_reviews.csv')

# Ambil kolom penting (sesuaikan kalau nama kolom beda)
df_gojek = df_gojek[['content','score']]
df_grab = df_grab[['content','score']]

# Tambahkan penanda aplikasi
df_gojek['app'] = 'Gojek'
df_grab['app'] = 'Grab'

print("Jumlah data Gojek:", len(df_gojek))
print("Jumlah data Grab:", len(df_grab))

Jumlah data Gojek: 5000
Jumlah data Grab: 5000


In [ ]:
# Gabung Dataset
df = pd.concat([df_gojek, df_grab], ignore_index=True)

print("Total data gabungan:", len(df))
df.head()

Total data gabungan: 10000


,content,score,app
0,Mantap,5,Gojek
1,pesanan cepat di terima,5,Gojek
2,penyiksa driver,1,Gojek
3,mudah digunakan,5,Gojek
4,mantap,5,Gojek


In [ ]:
# 3. BUAT LABEL DARI RATING
def label_sentiment(score):
    if score >= 4:
        return "positif"
    elif score == 3:
        return "netral"
    else:
        return "negatif"

df['label'] = df['score'].apply(label_sentiment)

print(df['label'].value_counts())

label
positif    6561
negatif    3053
netral      386
Name: count, dtype: int64


In [ ]:
# 4. CLEANING TEXT
def clean_text(text):
    text = text.lower()  # case folding
    text = re.sub(r"http\S+", "", text)  # hapus link
    text = re.sub(r"@\w+", "", text)  # hapus mention
    text = re.sub(r"#\w+", "", text)  # hapus hashtag
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # hapus angka & simbol
    text = re.sub(r"\s+", " ", text).strip()  # hapus spasi berlebih
    return text

df['clean_review'] = df['content'].apply(clean_text)

In [ ]:
# 5. TOKENIZATION
nltk.download('punkt_tab') # Download the missing 'punkt_tab' resource
df['tokens'] = df['clean_review'].apply(word_tokenize)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
# 6. STOPWORD REMOVAL
factory = StopWordRemoverFactory()
stopwords = factory.get_stop_words()

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stopwords]

df['filtered_tokens'] = df['tokens'].apply(remove_stopwords)

# Gabungkan kembali jadi kalimat
df['final_text'] = df['filtered_tokens'].apply(lambda x: " ".join(x))

df[['content', 'final_text']].head()

,content,final_text
0,Mantap,mantap
1,pesanan cepat di terima,pesanan cepat terima
2,penyiksa driver,penyiksa driver
3,mudah digunakan,mudah digunakan
4,mantap,mantap


In [ ]:
# 7. ENCODE LABEL
label_encoder = LabelEncoder()
df['encoded_label'] = label_encoder.fit_transform(df['label'])

print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

{'negatif': np.int64(0), 'netral': np.int64(1), 'positif': np.int64(2)}


In [ ]:
# 8. TOKENIZER (TEXT TO SEQUENCE)
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['final_text'])

sequences = tokenizer.texts_to_sequences(df['final_text'])

In [ ]:
# 9. PADDING
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

X = padded_sequences
y = df['encoded_label']

print("Shape X:", X.shape)
print("Shape y:", y.shape)

Shape X: (10000, 100)
Shape y: (10000,)


In [ ]:
# 10. SPLIT DATASET
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 7000
Validation: 1500
Test: 1500


In [ ]:
# SAVE DATASET CLEAN
df_clean = df[['final_text', 'label', 'encoded_label']]

df_clean.to_csv("dataset_clean.csv", index=False)

print("Dataset bersih berhasil disimpan!")

Dataset bersih berhasil disimpan!
